# Day H4 — 임상적 유용성: 결정 곡선과 순 이득

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/H-clinical-prediction/day-H4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/H-clinical-prediction/day-H4.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — numpy로 순 이득 계산하기

같은 가상 모집단에서 세 가지 모델을 비교합니다. "good"은 개발 자료로 맞춘 로지스틱 회귀, "over"는 순서는 같지만 위험을 크게 부풀린 버전(다른 집단에서 개발된 모델을 그대로 가져온 상황을 흉내), "weak"는 판별도 약하고 보정도 어긋난 버전입니다.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(5)
def make(n):
    X = rng.normal(size=(n, 3))
    y = rng.random(n) < 1 / (1 + np.exp(-(-2.0 + X @ [1.0, 0.7, 0.4])))   # 유병률 약 20%
    return X, y.astype(int)

X_dev, y_dev = make(1000)
X_te, y_te = make(2000)
p_good = LogisticRegression().fit(X_dev, y_dev).predict_proba(X_te)[:, 1]
lp = np.log(p_good / (1 - p_good))
p_over = 1 / (1 + np.exp(-(lp + 1.5)))          # 같은 순서, 위험을 크게 부풀린 버전(다른 기관에서 개발된 모델이라 가정)
p_weak = 1 / (1 + np.exp(-(0.3 * lp - 1.0)))    # 판별도 약하고 보정도 어긋난 버전에 잡음을 더함
p_weak = np.clip(p_weak + rng.normal(0, 0.05, len(p_weak)), 0.001, 0.999)

def net_benefit(y, p, pt):
    """임계 확률 pt에서 '양성이면 다음 검사를 한다'는 전략의 순 이득."""
    n = len(y)
    pos = p >= pt
    tp = np.sum(pos & (y == 1)); fp = np.sum(pos & (y == 0))
    return tp / n - fp / n * pt / (1 - pt)

def treat_all(y, pt):
    prev = y.mean()
    return prev - (1 - prev) * pt / (1 - pt)

print(f"유병률 {y_te.mean():.3f} | AUC good {roc_auc_score(y_te, p_good):.3f}, "
      f"over {roc_auc_score(y_te, p_over):.3f}, weak {roc_auc_score(y_te, p_weak):.3f}")
print("  pt  | 모두 검사  아무도 안 함 | good    over    weak")
for pt in [0.05, 0.10, 0.20, 0.30, 0.40]:
    print(f" {pt:.2f} |  {treat_all(y_te, pt):+.3f}     +0.000    | "
          f"{net_benefit(y_te, p_good, pt):+.3f}  {net_benefit(y_te, p_over, pt):+.3f}  {net_benefit(y_te, p_weak, pt):+.3f}")

### 예시 2 — 보정만 고치면 순 이득이 돌아온다

예시 1 다음 셀에 이어서 실행합니다. 과대 예측 모델을, 시험에 쓰지 않은 별도 자료 500명으로 절편만 재보정합니다([Day E6](../E-ehr/day-E6.md)의 재보정).

In [ ]:
# 예시 2 — 과대 예측 모델을 별도 자료 500명으로 절편만 재보정하면
import statsmodels.api as sm
model_good = LogisticRegression().fit(X_dev, y_dev)
X_cal, y_cal = make(500)
q = model_good.predict_proba(X_cal)[:, 1]
lp_cal = np.log(q / (1 - q)) + 1.5            # 재보정용 자료에서 과대 예측 모델이 내놓는 logit
shift = sm.GLM(y_cal, np.ones((500, 1)), family=sm.families.Binomial(), offset=lp_cal).fit().params[0]
p_fixed = 1 / (1 + np.exp(-(np.log(p_over / (1 - p_over)) + shift)))
print(f"추정한 절편 보정량 {shift:+.2f} | 예측 평균: 과대 {p_over.mean():.3f} → 재보정 {p_fixed.mean():.3f} (관찰 {y_te.mean():.3f})")
for pt in [0.10, 0.20, 0.30, 0.40]:
    print(f" pt {pt:.2f} | over {net_benefit(y_te, p_over, pt):+.3f} → 재보정 {net_benefit(y_te, p_fixed, pt):+.3f} | good {net_benefit(y_te, p_good, pt):+.3f}")

## 직접 해보기

1. `notebooks/advanced/H/H4_decision_curve.ipynb`에서 예시 1–2를 실행합니다.
2. 결정 곡선을 그립니다. 예시 1 다음 셀에 붙여 넣습니다.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

out = Path("../../../figures/advanced/H"); out.mkdir(parents=True, exist_ok=True)
pts = np.arange(0.01, 0.61, 0.01)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(pts, [treat_all(y_te, t) for t in pts], color="gray", label="treat all")
ax.axhline(0, color="black", lw=1, label="treat none")
for name, q in [("calibrated model", p_good), ("over-predicting (same AUC)", p_over), ("weak model", p_weak)]:
    ax.plot(pts, [net_benefit(y_te, q, t) for t in pts], label=name)
ax.set_ylim(-0.05, y_te.mean() + 0.02)
ax.set_xlabel("threshold probability"); ax.set_ylabel("net benefit")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(out / "H4_decision_curve.png", dpi=150)
print("saved", out / "H4_decision_curve.png")